In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
INPUT_FILE = Path("../data/02_intermediate/adni_merged_imputed.csv")
OUTPUT_FILE = Path("../data/02_intermediate/adni_cleaned_feature_selected.csv")

In [3]:
# 1. Define key MRI features
KEY_MRI_FEATURES = [
    "Left.Hippocampus_combat", "Right.Hippocampus_combat",
    "lh_entorhinal_volume_combat", "rh_entorhinal_volume_combat",
    "lh_entorhinal_thickness_combat", "rh_entorhinal_thickness_combat",
    "Left.Lateral.Ventricle_combat", "Right.Lateral.Ventricle_combat",
    "BrainSegVol_combat", "EstimatedTotalIntraCranialVol_combat"
]

In [4]:
# 2. Define core features
CORE_FEATURES = [
    "RID", "PTID", "VISCODE", "EXAMDATE", 
    "DIAGNOSIS", # 1=CN, 2=MCI, 3=AD
    "GENOTYPE", 
    "pT217_F", "AB42_F", "AB40_F", "NfL_Q", "GFAP_Q",
    "PHC_MEM", "PHC_EXF", "PHC_LAN"
]

In [5]:
def calculate_months_from_baseline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates precise months from baseline using actual dates, 
    then filters for the 36 month cap
    """
    # Create a deep copy to prevent SettingWithCopyWarning
    df = df.copy()
    
    # Solid date format
    df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'], errors='coerce')
    df = df.dropna(subset=['EXAMDATE'])

    # Find baseline date for each patient
    # We take the minimum date for each patient as their 'Day 0'
    baseline_dates = df.groupby('RID')['EXAMDATE'].transform('min')
    
    # Calculate difference in days
    df['Days_From_Baseline'] = (df['EXAMDATE'] - baseline_dates).dt.days
    
    # Convert to months and round to nearest integer
    df['Month'] = (df['Days_From_Baseline'] / 30.44).round().astype(int)
    
    # 36 MONTH CAP
    initial_count = len(df)
    df = df[df['Month'] <= 36]
    dropped_count = initial_count - len(df)
    
    print(f"36 Month Cap: Dropped {dropped_count} rows beyond 3 years.")
    print(f"Remaining Data Points: {len(df)}")
    
    return df

In [6]:
def feature_selection_and_imputation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Selects columns and imputes missing MRI data using DG medians
    """    
    # 1. Select Columns
    available_mri = [c for c in KEY_MRI_FEATURES if c in df.columns]
    
    # 2: Explicitly include month in the features to keep
    cols_to_keep = CORE_FEATURES + available_mri + ['Month']
    
    df_clean = df[cols_to_keep].copy()
    
    print(f"Reduced feature set to {df_clean.shape[1]} columns.")
    
    # 2. Impute missing MRI features
    for col in available_mri:
        missing_count = df_clean[col].isnull().sum()
        if missing_count > 0:
            # Impute by Diagnosis group which will help retain signal
            df_clean[col] = df_clean[col].fillna(
                df_clean.groupby('DIAGNOSIS')[col].transform('median')
            )
            # Fallback is the global median
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            
            print(f"Imputed {missing_count} missing values in {col}")
            
    return df_clean

In [ ]:
def main():
    if not INPUT_FILE.exists():
        print(f"Can't find file at {INPUT_FILE}")
        return
        
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    print(f"Loaded Data: {df.shape}")
    
    # 1. Calculate time & cap at 36 months
    df = calculate_months_from_baseline(df)
    
    # 2. Clean & impute
    df_final = feature_selection_and_imputation(df)
    
    # 3. Final Inspection
    print(f"Shape: {df_final.shape}")
    print(f"Unique Months: {sorted(df_final['Month'].unique())}")
    
    # 3: Add patient count
    unique_patients = df_final['RID'].nunique()
    print(f"Total unique patients based on RID: {unique_patients}")
    
    # Check for NaNs
    missing_total = df_final.isnull().sum().sum()
    print(f"Total missing values: {missing_total}")
    
    # 4. Save
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved cleaned dataset to: {OUTPUT_FILE}")

In [8]:
if __name__ == "__main__":
    main()

Loaded Data: (15989, 224)
36 Month Cap: Dropped 4286 rows beyond 3 years.
Remaining Data Points: 11631
Reduced feature set to 25 columns.
Imputed 6043 missing values in Left.Hippocampus_combat
Imputed 6043 missing values in Right.Hippocampus_combat
Imputed 6043 missing values in lh_entorhinal_volume_combat
Imputed 6043 missing values in rh_entorhinal_volume_combat
Imputed 6043 missing values in lh_entorhinal_thickness_combat
Imputed 6043 missing values in rh_entorhinal_thickness_combat
Imputed 6043 missing values in Left.Lateral.Ventricle_combat
Imputed 6043 missing values in Right.Lateral.Ventricle_combat
Imputed 6043 missing values in BrainSegVol_combat
Imputed 6043 missing values in EstimatedTotalIntraCranialVol_combat
Shape: (11631, 25)
Unique Months: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.in